# Pretrained CNN and Vision Transformer on the Same Dataset

## Objective

The goal of this notebook is to compare a pretrained convolutional neural network and a pretrained vision transformer on the same image classification dataset. The workflow keeps the same spirit as `model_comparison_v1.ipynb`, but the models now start from ImageNet pretrained weights so training converges faster.


## Introduction

This notebook follows a complete image classification workflow. I prepare the dataset, adapt two pretrained backbones to the target number of classes, fine-tune them under a shared setting, save the best checkpoints, reload them, and compare their predictions on the same custom images.

Compared with training from scratch, pretrained models usually learn faster and reach stronger results with fewer epochs, which is useful for local experiments and homework-style comparisons.


## Why Compare a Pretrained CNN and a Pretrained ViT

A pretrained CNN already has strong local feature extractors from large-scale image data. A pretrained vision transformer already has global token-based representations from large-scale pretraining. Placing them in the same notebook makes it easier to observe how convolution-based inductive bias and transformer-style representation behave under the same downstream task.


## 1. Import the libraries


In [ ]:
import csv
import copy
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torchvision.models import EfficientNet_B0_Weights, ViT_B_16_Weights


## 2. Reproducibility and runtime


In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

runtime_mode = 'colab'  # 'local' or 'colab'


In [ ]:
if runtime_mode == 'colab':
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    project_root = Path('/content/drive/MyDrive/classification-deep-learning-project')
    data_dir = Path('/content/data')
    torch_cache_dir = Path('/content/torch-cache')
else:
    project_root = Path.cwd()
    data_dir = project_root / 'data'
    torch_cache_dir = project_root / '.torch-cache'

model_dir = project_root / 'models'
prediction_dir = project_root / 'predictions'
test_images_dir = project_root / 'test_images'
custom_dataset_dir = project_root / 'custom_dataset'

data_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)
prediction_dir.mkdir(parents=True, exist_ok=True)
test_images_dir.mkdir(parents=True, exist_ok=True)
custom_dataset_dir.mkdir(parents=True, exist_ok=True)
torch_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(torch_cache_dir)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin_memory = device.type == 'cuda'

print('Runtime mode =', runtime_mode)
print('project_root =', project_root)
print('data_dir =', data_dir)
print('torch_cache_dir =', torch_cache_dir)
print('device =', device)
if device.type == 'cuda':
    print('gpu =', torch.cuda.get_device_name(0))


## 3. Choose the dataset and the common parameters


In [ ]:
train_dataset_name = 'cifar10'  # 'cifar10', 'cifar100', or 'custom_folder'
test_dataset_name = 'cifar10'   # 'cifar10', 'cifar100', or 'custom_folder'
image_size = 224
cnn_batch_size = 64
vit_batch_size = 32
num_epochs = 5
head_only_epochs = 1
learning_rate = 3e-4
backbone_learning_rate = 8e-5
weight_decay = 1e-4
num_workers = 2
quick_subset_train = None  # example: 5000
quick_subset_test = None   # example: 1000

cnn_name = 'efficientnet_b0'
vit_name = 'vit_b_16'


## 4. Prepare the shared dataset


In [ ]:
cnn_weights = EfficientNet_B0_Weights.DEFAULT
vit_weights = ViT_B_16_Weights.DEFAULT

cnn_train_transform = cnn_weights.transforms()
cnn_test_transform = cnn_weights.transforms()
vit_train_transform = vit_weights.transforms()
vit_test_transform = vit_weights.transforms()

def maybe_subset(dataset, subset_size):
    if subset_size is None or subset_size >= len(dataset):
        return dataset
    indices = list(range(subset_size))
    return torch.utils.data.Subset(dataset, indices)

def build_split(dataset_name, split_name, transform):
    if dataset_name == 'cifar10':
        is_train = split_name == 'train'
        dataset = datasets.CIFAR10(root=str(data_dir), train=is_train, download=True, transform=transform)
        return dataset, dataset.classes
    if dataset_name == 'cifar100':
        is_train = split_name == 'train'
        dataset = datasets.CIFAR100(root=str(data_dir), train=is_train, download=True, transform=transform)
        return dataset, dataset.classes
    if dataset_name == 'custom_folder':
        folder_name = 'train' if split_name == 'train' else 'val'
        dataset = datasets.ImageFolder(root=str(custom_dataset_dir / folder_name), transform=transform)
        return dataset, dataset.classes
    raise ValueError('Unsupported dataset name.')

cnn_trainset, train_classes = build_split(train_dataset_name, 'train', cnn_train_transform)
vit_trainset, vit_train_classes = build_split(train_dataset_name, 'train', vit_train_transform)
cnn_testset, test_classes = build_split(test_dataset_name, 'test', cnn_test_transform)
vit_testset, vit_test_classes = build_split(test_dataset_name, 'test', vit_test_transform)

if train_classes != vit_train_classes:
    raise ValueError('CNN and ViT training classes do not match.')
if test_classes != vit_test_classes:
    raise ValueError('CNN and ViT test classes do not match.')
if train_classes != test_classes:
    raise ValueError('Training classes and test classes do not match. Use datasets with the same class order.')

classes = train_classes

cnn_trainset = maybe_subset(cnn_trainset, quick_subset_train)
cnn_testset = maybe_subset(cnn_testset, quick_subset_test)
vit_trainset = maybe_subset(vit_trainset, quick_subset_train)
vit_testset = maybe_subset(vit_testset, quick_subset_test)

num_classes = len(classes)

cnn_trainloader = DataLoader(cnn_trainset, batch_size=cnn_batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
cnn_testloader = DataLoader(cnn_testset, batch_size=cnn_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
vit_trainloader = DataLoader(vit_trainset, batch_size=vit_batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
vit_testloader = DataLoader(vit_testset, batch_size=vit_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

experiment_name = f'{train_dataset_name}_train_{test_dataset_name}_test'

print('train_dataset_name =', train_dataset_name)
print('test_dataset_name =', test_dataset_name)
print('classes =', classes)
print('num_classes =', num_classes)
print('cnn train size =', len(cnn_trainset), 'cnn test size =', len(cnn_testset))
print('vit train size =', len(vit_trainset), 'vit test size =', len(vit_testset))


## 5. Build the pretrained models


In [ ]:
def build_pretrained_cnn(num_classes):
    model = models.efficientnet_b0(weights=cnn_weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


def build_pretrained_vit(num_classes):
    model = models.vit_b_16(weights=vit_weights)
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)
    return model


def freeze_backbone(model, head_keywords):
    for name, parameter in model.named_parameters():
        parameter.requires_grad = any(keyword in name for keyword in head_keywords)


def unfreeze_all(model):
    for parameter in model.parameters():
        parameter.requires_grad = True


cnn_head_keywords = ['classifier']
vit_head_keywords = ['heads']

cnn_model = build_pretrained_cnn(num_classes).to(device)
vit_model = build_pretrained_vit(num_classes).to(device)


In [ ]:
print(cnn_model)
print(vit_model)


## 6. Training helpers


In [ ]:
def evaluate_model(model, dataloader, criterion):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)
            running_correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100.0 * running_correct / total


def create_optimizer(model):
    head_parameters = []
    backbone_parameters = []

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if 'classifier' in name or 'heads' in name:
            head_parameters.append(parameter)
        else:
            backbone_parameters.append(parameter)

    parameter_groups = []
    if backbone_parameters:
        parameter_groups.append({'params': backbone_parameters, 'lr': backbone_learning_rate})
    if head_parameters:
        parameter_groups.append({'params': head_parameters, 'lr': learning_rate})

    return optim.AdamW(parameter_groups, weight_decay=weight_decay)


def train_model(model, trainloader, testloader, criterion, model_name, head_keywords):
    history = []
    best_state_dict = None
    best_accuracy = 0.0

    freeze_backbone(model, head_keywords)
    optimizer = create_optimizer(model)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_model_path = model_dir / f'{experiment_name}_{model_name}_pretrained_best.pth'
    last_model_path = model_dir / f'{experiment_name}_{model_name}_pretrained_last.pth'

    for epoch in range(num_epochs):
        if epoch == head_only_epochs:
            unfreeze_all(model)
            optimizer = create_optimizer(model)
            scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, num_epochs - epoch))

        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0
        start_time = time.time()

        for inputs, labels in trainloader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)
            running_correct += (predictions == labels).sum().item()
            total += labels.size(0)

        scheduler.step()

        train_loss = running_loss / total
        train_accuracy = 100.0 * running_correct / total
        validation_loss, validation_accuracy = evaluate_model(model, testloader, criterion)
        epoch_time = time.time() - start_time

        history.append({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_accuracy': train_accuracy,
            'validation_loss': validation_loss,
            'validation_accuracy': validation_accuracy,
            'epoch_time_seconds': epoch_time,
        })

        print(
            f'{model_name} epoch {epoch + 1}/{num_epochs} | '
            f'train loss {train_loss:.4f} | train acc {train_accuracy:.2f}% | '
            f'val loss {validation_loss:.4f} | val acc {validation_accuracy:.2f}% | '
            f'time {epoch_time:.1f}s'
        )

        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save(best_state_dict, best_model_path)

    torch.save(model.state_dict(), last_model_path)

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return history, best_model_path, last_model_path, best_accuracy


## 7. Train the pretrained CNN


In [ ]:
cnn_criterion = nn.CrossEntropyLoss()
cnn_history, cnn_best_model_path, cnn_last_model_path, cnn_best_accuracy = train_model(
    model=cnn_model,
    trainloader=cnn_trainloader,
    testloader=cnn_testloader,
    criterion=cnn_criterion,
    model_name=cnn_name,
    head_keywords=cnn_head_keywords,
)


## 8. Train the pretrained vision transformer


In [ ]:
vit_criterion = nn.CrossEntropyLoss()
vit_history, vit_best_model_path, vit_last_model_path, vit_best_accuracy = train_model(
    model=vit_model,
    trainloader=vit_trainloader,
    testloader=vit_testloader,
    criterion=vit_criterion,
    model_name=vit_name,
    head_keywords=vit_head_keywords,
)


## 9. Save summary


In [ ]:
print('CNN best model:', cnn_best_model_path)
print('CNN last model:', cnn_last_model_path)
print('ViT best model:', vit_best_model_path)
print('ViT last model:', vit_last_model_path)
print('CNN best validation accuracy:', f'{cnn_best_accuracy:.2f}%')
print('ViT best validation accuracy:', f'{vit_best_accuracy:.2f}%')


## 10. Compare the validation curves


In [ ]:
epochs_axis = [item['epoch'] for item in cnn_history]

cnn_val_acc = [item['validation_accuracy'] for item in cnn_history]
vit_val_acc = [item['validation_accuracy'] for item in vit_history]
cnn_val_loss = [item['validation_loss'] for item in cnn_history]
vit_val_loss = [item['validation_loss'] for item in vit_history]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_axis, cnn_val_loss, marker='o', label='CNN Validation Loss')
plt.plot(epochs_axis, vit_val_loss, marker='o', label='ViT Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Validation Loss Comparison')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_axis, cnn_val_acc, marker='o', label='CNN Validation Accuracy')
plt.plot(epochs_axis, vit_val_acc, marker='o', label='ViT Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Validation Accuracy Comparison')
plt.legend()
plt.tight_layout()
plt.show()


## 11. Reload the best models


In [ ]:
cnn_model = build_pretrained_cnn(num_classes).to(device)
cnn_model.load_state_dict(torch.load(cnn_best_model_path, map_location=device))
cnn_model.eval()

vit_model = build_pretrained_vit(num_classes).to(device)
vit_model.load_state_dict(torch.load(vit_best_model_path, map_location=device))
vit_model.eval()


## 12. Predict the same custom images with both models


In [ ]:
image_paths = sorted(
    [
        path
        for path in test_images_dir.iterdir()
        if path.is_file() and path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']
    ]
)
print('Number of test images found:', len(image_paths))

comparison_prediction_file = prediction_dir / f'{experiment_name}_{cnn_name}_{vit_name}_pretrained_comparison.csv'

with comparison_prediction_file.open('w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['image_name', 'cnn_prediction', 'cnn_confidence', 'vit_prediction', 'vit_confidence'])

    for image_path in image_paths:
        image = Image.open(image_path).convert('RGB')
        cnn_tensor = cnn_test_transform(image).unsqueeze(0).to(device)
        vit_tensor = vit_test_transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            cnn_output = cnn_model(cnn_tensor)
            vit_output = vit_model(vit_tensor)

        cnn_probabilities = torch.softmax(cnn_output, dim=1)
        vit_probabilities = torch.softmax(vit_output, dim=1)

        cnn_confidence, cnn_prediction = torch.max(cnn_probabilities, dim=1)
        vit_confidence, vit_prediction = torch.max(vit_probabilities, dim=1)

        writer.writerow([
            image_path.name,
            classes[cnn_prediction.item()],
            float(cnn_confidence.item()),
            classes[vit_prediction.item()],
            float(vit_confidence.item()),
        ])

print('Prediction file saved to:', comparison_prediction_file)


## 13. Confusion matrices


In [ ]:
def collect_predictions(model, dataloader):
    predictions_all = []
    labels_all = []

    model.eval()
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device, non_blocking=True)
            outputs = model(inputs)
            predictions = outputs.argmax(dim=1).cpu()
            predictions_all.extend(predictions.tolist())
            labels_all.extend(labels.tolist())

    confusion = torch.zeros(num_classes, num_classes, dtype=torch.int64)
    for true_label, predicted_label in zip(labels_all, predictions_all):
        confusion[true_label, predicted_label] += 1

    return confusion, predictions_all, labels_all


cnn_confusion, cnn_predictions_all, labels_all = collect_predictions(cnn_model, cnn_testloader)
vit_confusion, vit_predictions_all, _ = collect_predictions(vit_model, vit_testloader)


In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(cnn_confusion.numpy(), cmap='Blues')
plt.title('Pretrained CNN Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(range(num_classes), classes, rotation=90)
plt.yticks(range(num_classes), classes)
plt.colorbar(fraction=0.046, pad=0.04)

plt.subplot(1, 2, 2)
plt.imshow(vit_confusion.numpy(), cmap='Greens')
plt.title('Pretrained ViT Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(range(num_classes), classes, rotation=90)
plt.yticks(range(num_classes), classes)
plt.colorbar(fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


## 14. Parameter count comparison


In [ ]:
cnn_num_params = sum(parameter.numel() for parameter in cnn_model.parameters())
vit_num_params = sum(parameter.numel() for parameter in vit_model.parameters())

print('Pretrained CNN number of parameters:', cnn_num_params)
print('Pretrained ViT number of parameters:', vit_num_params)


## 15. Agreement and disagreement analysis


In [ ]:
both_correct = 0
cnn_only_correct = 0
vit_only_correct = 0
both_wrong = 0

for true_label, cnn_prediction, vit_prediction in zip(labels_all, cnn_predictions_all, vit_predictions_all):
    cnn_correct = cnn_prediction == true_label
    vit_correct = vit_prediction == true_label

    if cnn_correct and vit_correct:
        both_correct += 1
    elif cnn_correct and not vit_correct:
        cnn_only_correct += 1
    elif not cnn_correct and vit_correct:
        vit_only_correct += 1
    else:
        both_wrong += 1

total_samples = len(labels_all)
print('Both correct:', both_correct, f'({100.0 * both_correct / total_samples:.2f}%)')
print('Only CNN correct:', cnn_only_correct, f'({100.0 * cnn_only_correct / total_samples:.2f}%)')
print('Only ViT correct:', vit_only_correct, f'({100.0 * vit_only_correct / total_samples:.2f}%)')
print('Both wrong:', both_wrong, f'({100.0 * both_wrong / total_samples:.2f}%)')


## Results and Discussion

The main purpose of this notebook is to compare two pretrained transfer learning strategies under the same downstream setting. Because both backbones start from ImageNet weights, the learning curves are expected to improve faster than in the from-scratch notebook. This makes the experiment more practical when running locally, while still preserving a meaningful CNN-versus-transformer comparison.

The confusion matrices and agreement analysis help go beyond overall accuracy. They show whether the CNN and ViT succeed on the same categories, whether one model corrects errors made by the other, and whether the pretrained representations transfer equally well to small natural-image datasets such as CIFAR.


## Conclusion

In this notebook, I compared a pretrained CNN and a pretrained vision transformer on the same dataset under a shared workflow. The experiment keeps the structure of the previous notebook but replaces training from scratch with transfer learning, which is faster and usually more stable on a local machine.

This notebook can also be extended easily. For example, the subset settings can be enabled for a smoke test, the number of head-only epochs can be tuned, and other pretrained backbones from `torchvision.models` can be plugged into the same evaluation pipeline.


## References

1. Krizhevsky, Sutskever, and Hinton. *ImageNet Classification with Deep Convolutional Neural Networks*. 2012.
2. Dosovitskiy et al. *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*. 2021.
3. PyTorch Documentation. https://pytorch.org/
4. Torchvision Documentation. https://pytorch.org/vision/stable/models.html
